## Eksāmena darbā izmantotais kods

Pirmkārt, tiek instalētas nepieciešamās bibliotēkas. Darbam tiek izmantots Beautiful Soup datu izgūšanai no failiem un Stanza teksta vārdu semantisko lomu marķēšanai.

In [34]:
%pip install beautifulsoup4
%pip install stanza

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 22.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   -- ------------------------------------- 7.1/123.0 MB 33.4 MB/s eta 0:00:04
   ---- ----------------------------------- 12.8/123.0 MB 30.2 MB/s eta 0:00:04
   ------ --------------------------------- 19.1/123.0 MB 30.4 MB/s eta 0:00:04
   -------- ------------------------------- 26.7/123.0 MB 31.5 MB/s eta 0:00:04
   ----------- ---------------------------- 34.3/123.0 MB 32.3 MB/s eta 0:00:03
   ------------ --------------------------- 39.8/123.0 MB 31.5 MB/s eta 0:00:03
   -------------- ------------------------- 45.9/123.0 MB 31.0 MB/s eta 0:00:03
   ---------------- ----------------------- 52.2/123.0 MB 30.7 MB/s eta 0:00:03
   ------------------- -------------------- 58.7/123.0 MB 30.6 MB/s eta 0:00:0

Kad nepieciešami rīki ir instalēti, sāk ar datu izgūšanu no failiem, to satīrīšanu un salikšanu vienotā struktūrā.

In [ ]:
from bs4 import BeautifulSoup
import os
import re

# Papildfunkcija, kas skaitli pārvērš par divu simbolu garu virkni.
def twoString(x):
    if (x < 10):
        return "0" + str(x)
    else:
        return str(x)
# Papildfunkcija, kas skaitli pārvērš par trīs simbolu garu virkni.
def threeString(x):
    if (x < 10):
        return "00" + str(x)
    elif (x < 100):
        return "0" + str(x)
    else:
        return str(x)

directories = [x[0] for x in os.walk("text")] # Atrod visus ceļus uz dažādajām.

# Iziet cauri visiem sējumiem un no failiem izvelk tekstus, apvienojot tos vienotā struktūrā pēc attīrīšanas.
all_text = []
for directory in directories:
    if directory.count('\\') > 1:   # Tiek izņemtas visas mapes, kurās neglabājas faili.
        files = []
        for (dirpath, dirnames, filenames) in os.walk(directory):
            files.extend(filenames)
            break

        file_num = 1
        for file in files:    
            id = directory + '\\' + threeString(file_num)

            with open(directory+'\\'+file, encoding="windows-1257") as page:
                soup = BeautifulSoup(page.read(), "html.parser")
                extracted_paragraphs = soup.find_all('p')
                extracted_text = ""
                p_count = 0
                y_flag = False
                for p in extracted_paragraphs:
                    if p_count > 1:
                        temp_text = p.get_text()
                        if temp_text == "":
                            continue
                        if temp_text[0].isdigit():
                            continue
                        if "y" in temp_text:
                            y_flag = True
                            break
                        
                        # Teksta tīrīšana.
                        temp_text = re.sub('[ ]+', ' ', temp_text)
                        temp_text = temp_text.replace('\n', ' ')    # Tekstu apvieno vienā rindkopā.

                        temp_text = temp_text.replace('`', '\'')
                        temp_text = temp_text.replace('\'\'', '"')
                        temp_text = temp_text.replace('\'', '')

                        temp_text = temp_text.replace('„', '"')
                        temp_text = temp_text.replace('«', '"')
                        temp_text = temp_text.replace('»', '"')

                        temp_text = re.sub(r'\d+', "100", temp_text)

                        temp_text = temp_text.replace('[', '')
                        temp_text = temp_text.replace(']', '')
                        temp_text = temp_text.strip()
                        
                        if temp_text != "":
                            extracted_text = extracted_text + temp_text + ' '

                    p_count = p_count + 1

                if not y_flag:
                    all_text.append([id, extracted_text])
                file_num = file_num + 1

Pēc teksta iegūšanas un sakārtošanas notiek tā vārdu marķēšana, kā arī biežumu aprēķināšana.

In [92]:
import stanza
import json

pipeline = stanza.Pipeline(lang='lv', processors='tokenize,pos,lemma,depparse')
# print(*[f'id: {word.id}\tword: {word.text}\tlemma: {word.lemma}\txpos: {word.upos}\tdeprel: {word.deprel}' for sent in tagged_text.sentences for word in sent.words], sep='\n')

result_list = []
for i in range(20):
    tagged_text = pipeline(all_text[i][1])
    freq_count = tagged_text.num_words
    freq_list = {}
    subject_list = {}        # Uzglabā tieši teikuma priekšmetu biežumu.
    for sent in tagged_text.sentences:
        for word in sent.words:
            if word.upos == "NOUN" or word.upos == "PROPN":
                if word.deprel == "nsubj":
                    if word.lemma in subject_list:
                        subject_list[word.lemma] = subject_list[word.lemma] + 1
                    else:
                        subject_list[word.lemma] = 0
                
                if word.lemma in freq_list:
                    freq_list[word.lemma] = freq_list[word.lemma] + 1
                else:
                    freq_list[word.lemma] = 0

    word_list = set()
    target_count = 2         # Īsākos tekstos varonis parādīsies mazāk reižu, bet garākos tekstos varoņi parādās vairrākkārt.
    if freq_count > 2000:
        target_count = 5
    elif freq_count > 1000:
        target_count = 4
    elif freq_count > 400:
        target_count = 3

    for word in subject_list:
        if (freq_list[word] >= target_count) and (subject_list[word] / freq_list[word] > 1/3):
            word_list.add(word)
    
    result_object = { "text": all_text[i][1], "characters": list(word_list) }
    result_list.append(result_object)

with open("results.json", 'w') as file:
    json.dump(result_list, file, indent=4)


2026-06-17 19:12:56 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-06-17 19:12:56 INFO: Downloaded file to C:\Users\simon\AppData\Local\StanfordNLP\stanza\Cache\1.12.0\resources\resources.json
2026-06-17 19:12:57 INFO: Loading these models for language: lv (Latvian):
| Processor | Package       |
-----------------------------
| tokenize  | lvtb          |
| pos       | lvtb_nocharlm |
| lemma     | lvtb_nocharlm |
| depparse  | lvtb_nocharlm |

2026-06-17 19:12:57 INFO: Using device: cpu
2026-06-17 19:12:57 INFO: Loading: tokenize
2026-06-17 19:12:57 INFO: Loading: pos
2026-06-17 19:12:59 INFO: Loading: lemma
2026-06-17 19:13:00 INFO: Loading: depparse
2026-06-17 19:13:00 INFO: Done loading processors!


In [33]:
# TEST BLOCK

print(len(all_text))
print(all_text[0][1])
print(all_text[1][1])
print(all_text[2][1])
print(all_text[100][1])
print(all_text[200][1])
print(all_text[300][1])
print(all_text[500][1])
print(all_text[1000][1])
print(all_text[1500][1])

2850
Vecam zaldātam gadījās reiz, ar draugiem satiekoties, stipri iedzerties, ka vairs tanī vakarā nespēja uz kazarmu aiziet, bet palika ceļmalā guļot. Par to nu nekāda liela nelaime nebūtu bijusi, jo bija silts vasaras laiks, ja rītā agri nebūtu uz smatru jāiet. Nabaga zaldātiņš, rītā agri pamodies, steidzās uz māju, bet atrada, ka citi biedri pie darba aizgājuši, un nu zināja gan, kāda pirts gaidāma. Nelaimīgais savās bēdās staigāja gar jūrmalu un nezināja, ko darīt: Vai jūrā nāvi meklēt, jeb pie palkavnieka iet un savu sodu saņemt. Pēdīgi viņš apņēmās pie palkavnieka iet, lai tur ietu kā iedams. Pa ceļu ejot, tas satika zemnieku, kas veda govi uz tirgu. "Cik maksā tā govs?" zaldāts vaicāja. "Desmit rubļu!" zemnieks atbildēja. Zaldāts, turīgs vīrs būdams, nopirka šo govi ar to nodomu saviem biedriem atdot, lai tie no tās sev labu magariču sataisa un par savu nelaimīgo biedru Dievu lūdz. Tā nu zaldāts, govi pie rokas vezdams, devās uz kazarmu. Bet netālu no tā ieraudzīja vecu vīriņu a